# PointVisor – Point-Supervised Semantic Segmentation on DLRSD
**LandVisor Project Task Solution**

Implements partial Focal CE loss, simulates point annotations on real remote sensing data, trains a segmentation model, and runs experiments.

In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import random
from glob import glob
from tqdm import tqdm
import pandas as pd
from skimage.exposure import match_histograms

DATA_ROOT = "DLRSD"
IMAGE_DIR = os.path.join(DATA_ROOT, "Images")
LABEL_DIR = os.path.join(DATA_ROOT, "Labels")

In [2]:
# Verify paths
print("DATA_ROOT  :", DATA_ROOT)
print("IMAGE_DIR  :", IMAGE_DIR)
print("LABEL_DIR  :", LABEL_DIR)

# Define all possible image extensions
exts = ['*.jpg', '*.jpeg', '*.png', '*.tif', '*.tiff']
all_images = []

for ext in exts:
    # Search for both lowercase and uppercase versions
    all_images.extend(glob(os.path.join(IMAGE_DIR, "**", ext), recursive=True))
    all_images.extend(glob(os.path.join(IMAGE_DIR, "**", ext.upper()), recursive=True))

print(f"Total images found: {len(all_images)}")

if len(all_images) > 0:
    print("First 5 image paths:")
    for p in all_images[:5]:
        print("   ", p)
else:
    # If still 0, check what is actually in the directory
    print(f"\n[!] ALERT: No images found in {IMAGE_DIR}")
    print("Actual directory contents:", os.listdir(IMAGE_DIR)[:10])

DATA_ROOT  : DLRSD
IMAGE_DIR  : DLRSD\Images
LABEL_DIR  : DLRSD\Labels
Total images found: 4200
First 5 image paths:
    DLRSD\Images\agricultural\agricultural00.tif
    DLRSD\Images\agricultural\agricultural01.tif
    DLRSD\Images\agricultural\agricultural02.tif
    DLRSD\Images\agricultural\agricultural03.tif
    DLRSD\Images\agricultural\agricultural04.tif


In [3]:
class PartialFocalLoss(nn.Module):
    """Partial Focal CE Loss"""
    def __init__(self, gamma=2.0, alpha=0.25, ignore_index=255):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.ignore_index = ignore_index

    def forward(self, pred, target, mask):
        # pred: (B, C, H, W) logits
        # target: (B, H, W) long
        # mask: (B, H, W) float 0/1 (only labeled points = 1)
        ce = nn.functional.cross_entropy(pred, target, reduction='none', ignore_index=self.ignore_index)
        pt = torch.exp(-ce)
        focal = self.alpha * (1 - pt) ** self.gamma * ce
        masked = focal * mask
        return masked.sum() / (mask.sum() + 1e-8)   # average only over labeled points

In [4]:
class DLRSDPointDataset(Dataset):
    def __init__(self, image_files, points_per_class=5, reference_img_path=None):
        self.image_files = image_files
        self.points_per_class = points_per_class
        # Use the first image as a style reference if none provided
        self.reference_img_path = reference_img_path if reference_img_path else image_files[0]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
    
        # Safer label path implementation
        rel_path = os.path.relpath(img_path, IMAGE_DIR)
        base_name = os.path.splitext(rel_path)[0]
        label_path = os.path.join(LABEL_DIR, base_name + '.png')

        # Load Image
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Histogram Matching 
        # This ensures the 'picture style' is consistent across different remote sensing tiles
        ref_img = cv2.imread(self.reference_img_path)
        ref_img = cv2.cvtColor(ref_img, cv2.COLOR_BGR2RGB)
        img = match_histograms(img, ref_img, channel_axis=-1)

        # Preprocessing & Normalization
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))

        # Load Label
        label = cv2.imread(label_path, cv2.IMREAD_GRAYSCALE)
        if label is None:
            # Fallback if label is missing to prevent crash
            label = np.zeros((img.shape[1], img.shape[2]), dtype=np.int64)
        label = label.astype(np.int64)

        # Simulate point labels [cite: 7, 8]
        # Addressing the challenge where only point-based tagging is available
        point_target, point_mask = self.simulate_points(label)

        return torch.from_numpy(img), torch.from_numpy(label), point_target, point_mask

    def simulate_points(self, label_np):
        """Creates a sparse target and a binary mask for the pfCE loss [cite: 12, 14]"""
        H, W = label_np.shape
        # 255 is the ignore_index for standard CE [cite: 18]
        point_target = np.full((H, W), 255, dtype=np.int64)
        point_mask = np.zeros((H, W), dtype=np.float32)

        for c in range(17):  # DLRSD classes
            ys, xs = np.where(label_np == c)
            if len(ys) == 0: continue
            
            # Randomly sample 'n' points to simulate incomplete tagging 
            n = min(self.points_per_class, len(ys))
            idx = random.sample(range(len(ys)), n)
            
            # Map samples to the target and mask
            point_target[ys[idx], xs[idx]] = c
            point_mask[ys[idx], xs[idx]] = 1.0

        return point_target, point_mask

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = smp.Unet(encoder_name='resnet34', encoder_weights='imagenet', in_channels=3, classes=17).to(device)

criterion = PartialFocalLoss(gamma=2.0)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

def train_one_epoch(dataloader, criterion, is_focal):
    model.train()
    total_loss = 0.0
    num_batches = 0

    for img, full_label, point_target, point_mask in tqdm(dataloader, desc="Training"):
        img = img.to(device)
        point_target = point_target.to(device)
        point_mask = point_mask.to(device)

        optimizer.zero_grad()
        pred = model(img)

        if is_focal:
            loss = criterion(pred, point_target, point_mask)
        else:
            loss = criterion(pred, point_target)  # vanilla CE ignores 255 automatically

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    return total_loss / num_batches if num_batches > 0 else 0.0

In [ ]:
# Use only first 300 images for fast experiments, it can be increasd
# Robust file collection for DLRSD subfolder structure
image_files = []
exts = ['*.jpg', '*.jpeg', '*.png', '*.tif', '*.tiff']
for ext in exts:
    image_files.extend(glob(os.path.join(IMAGE_DIR, "**", ext), recursive=True))
    image_files.extend(glob(os.path.join(IMAGE_DIR, "**", ext.upper()), recursive=True))

print(f"Total images found: {len(image_files)}")

# Use first 30 images for fast testing (increase to 300–1000+ later)
all_images = image_files[:30]

results = []

for n_points in [5, 15, 30]:
    for use_focal in [True, False]:
        # FIX: Re-initialize model here so each experiment starts fresh
        model = smp.Unet(encoder_name='resnet34', encoder_weights='imagenet', in_channels=3, classes=17).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=1e-4)

        dataset = DLRSDPointDataset(all_images, points_per_class=n_points)
        loader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=0)

        if use_focal:
            criterion = PartialFocalLoss(gamma=2.0)
        else:
            criterion = nn.CrossEntropyLoss(ignore_index=255)

        for epoch in range(5):
            model.train()
            total_loss = 0.0
            num_batches = 0

            for img, _, point_target, point_mask in tqdm(loader, desc=f"Epoch {epoch+1}/5"):
                img = img.to(device)
                point_target = point_target.to(device)
                point_mask = point_mask.to(device)

                optimizer.zero_grad()
                pred = model(img)

                if use_focal:
                    loss = criterion(pred, point_target, point_mask)
                else:
                    # Check if there are any valid labels
                    valid_mask = (point_target != 255)
                    if valid_mask.sum() == 0:
                        continue  # skip this batch - no supervision
                    loss = criterion(pred, point_target) # ← no mask for vanilla CE

                loss.backward()
                optimizer.step()

                total_loss += loss.item()
                num_batches += 1

            avg_loss = total_loss / num_batches if num_batches > 0 else 0.0
            print(f"  Epoch {epoch+1}/5 - Loss: {avg_loss:.4f}")

        results.append({
            "points_per_class": n_points,
            "focal": use_focal,
            "final_loss": avg_loss
        })

df = pd.DataFrame(results)
print("\nResults:")
print(df.round(4))
df.to_csv("experiment_results.csv", index=False)